In [1]:
import re

In [3]:
text = """
On 12/03/2024, Mohamed Azab sent an email to support@company.com regarding Ticket ID: CS-45821.
The issue occurred at 09:45 AM on server srv-eu-03 with IP address 192.168.1.45.
A follow-up message was sent on 13/03/2024 at 21:07 PM.

Customer details:
Name: Mohamed A. Azab
Email: mohamed.azab@gmail.com
Backup Email: azab_support@outlook.co.uk
Phone: +7-921-555-0198

System logs:
[INFO] 2024-03-12 09:45:33 - User login successful
[WARN] 2024-03-12 09:47:10 - Multiple failed attempts from IP 10.0.0.12
[ERROR] 2024-03-12 09:49:55 - Transaction TXN-992134 failed
[INFO] 2024-03-13 21:07:01 - Ticket CS-45821 resolved

Payment summary:
Invoice INV-77821 | Amount: $1,250.50 | Status: PAID
Invoice INV-77822 | Amount: €899.99 | Status: PENDING

Notes:
Please review the logs before 15/03/2024.
Contact admin@security.local if the issue persists.
"""


In [43]:
dates_pattern = r"\b\d{2}/\d{2}/\d{4}\b"

dates = re.findall(dates_pattern, text)
print(dates)

['12/03/2024', '13/03/2024', '15/03/2024']


In [44]:
emails_pattern = r"\b[a-zA-Z0-9.%-+]+@[a-zA-Z]+\.[a-zA-Z]{2,}\b"

emails = re.findall(emails_pattern, text)
print(emails)

['support@company.com', 'mohamed.azab@gmail.com', 'admin@security.local']


In [45]:
phones_pattern = r"\+\d{1,3}-\d{3}-\d{3}-\d{4}"
phones = re.findall(phones_pattern, text)

print(phones)

['+7-921-555-0198']


In [46]:
ips_pattern = r"\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b"

ips = re.findall(ips_pattern, text)
print(ips)

['192.168.1.45', '10.0.0.12']


In [47]:
tickets_pattern = r"\bCS-\d{5}\b"

tickets = re.findall(tickets_pattern, text)
print(tickets)

['CS-45821', 'CS-45821']


In [52]:
invoice_pattern = (
    r"Invoice\s(?P<invoice_id>INV-\d+)\s\|\s"
    r"Amount:\s(?P<amount>[$€][\d,]+\.\d{2})\s\|\s"
    r"Status:\s(?P<status>[A-Z]+)"
)

invoices = re.finditer(invoice_pattern, text)
print(invoices)
invoice_records = []

for match in invoices:
    invoice_records.append(match.groupdict())

print(invoice_records)

[{'invoice_id': 'INV-77821', 'amount': '$1,250.50', 'status': 'PAID'}, {'invoice_id': 'INV-77822', 'amount': '€899.99', 'status': 'PENDING'}]


In [53]:
log_pattern = (
    r"\[(?P<level>INFO|WARN|ERROR)\]\s"
    r"(?P<date>\d{4}-\d{2}-\d{2})\s"
    r"(?P<time>\d{2}:\d{2}:\d{2})\s-\s"
    r"(?P<message>.+)"
)

logs = re.finditer(log_pattern, text)

log_records = []
for log in logs:
    log_records.append(log.groupdict())

print(log_records)

[{'level': 'INFO', 'date': '2024-03-12', 'time': '09:45:33', 'message': 'User login successful'}, {'level': 'WARN', 'date': '2024-03-12', 'time': '09:47:10', 'message': 'Multiple failed attempts from IP 10.0.0.12'}, {'level': 'ERROR', 'date': '2024-03-12', 'time': '09:49:55', 'message': 'Transaction TXN-992134 failed'}, {'level': 'INFO', 'date': '2024-03-13', 'time': '21:07:01', 'message': 'Ticket CS-45821 resolved'}]


In [54]:
etl_output = {
    "dates": dates,
    "emails": emails,
    "ips": ips,
    "tickets": tickets,
    "phones": phones,
    "invoices": invoice_records,
    "logs": log_records
}
etl_output

{'dates': ['12/03/2024', '13/03/2024', '15/03/2024'],
 'emails': ['support@company.com',
  'mohamed.azab@gmail.com',
  'admin@security.local'],
 'ips': ['192.168.1.45', '10.0.0.12'],
 'tickets': ['CS-45821', 'CS-45821'],
 'phones': ['+7-921-555-0198'],
 'invoices': [{'invoice_id': 'INV-77821',
   'amount': '$1,250.50',
   'status': 'PAID'},
  {'invoice_id': 'INV-77822', 'amount': '€899.99', 'status': 'PENDING'}],
 'logs': [{'level': 'INFO',
   'date': '2024-03-12',
   'time': '09:45:33',
   'message': 'User login successful'},
  {'level': 'WARN',
   'date': '2024-03-12',
   'time': '09:47:10',
   'message': 'Multiple failed attempts from IP 10.0.0.12'},
  {'level': 'ERROR',
   'date': '2024-03-12',
   'time': '09:49:55',
   'message': 'Transaction TXN-992134 failed'},
  {'level': 'INFO',
   'date': '2024-03-13',
   'time': '21:07:01',
   'message': 'Ticket CS-45821 resolved'}]}

In [55]:
import csv

In [56]:
with open("log2.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=["level", "date", "time", "message"])
    writer.writeheader()
    writer.writerows(log_records)